# SPECTER2 embeddings on Google Colab

This notebook reads the prepared `colab_specter2_bundle` from Google Drive and generates one 768-dimensional SPECTER2 embedding for every paper.

Before running:

1. In Colab, select **Runtime → Change runtime type → GPU**.
2. Change `BUNDLE_ROOT` in the configuration cell.
3. Run all cells.

The notebook checkpoints every completed shard to Google Drive. A repeated run skips completed shards unless `OVERWRITE_COMPLETED` is set to `True`.

The final cell terminates the Colab runtime only after all expected shards pass validation.

In [1]:
%pip install -q -U adapters pyarrow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.1 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount("/content/drive")

MessageError: Error: credential propagation was unsuccessful

## Configuration

`BUNDLE_ROOT` must point to the uploaded `colab_specter2_bundle` directory.

In [ ]:
from pathlib import Path

BUNDLE_ROOT = Path(
    "/content/drive/MyDrive/REPLACE_WITH_YOUR_PATH/colab_specter2_bundle"
)

BATCH_SIZE = 64
USE_FP16 = True
OVERWRITE_COMPLETED = False
AUTO_TERMINATE = True
TERMINATION_DELAY_SECONDS = 15

LOCAL_SCRATCH = Path("/content/specter2_scratch")

In [ ]:
import gc
import json
import os
import platform
import shutil
import time
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm

if "REPLACE_WITH_YOUR_PATH" in str(BUNDLE_ROOT):
    raise ValueError("Set BUNDLE_ROOT to the Google Drive path of colab_specter2_bundle.")

if not BUNDLE_ROOT.exists():
    raise FileNotFoundError(BUNDLE_ROOT)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is available. Select a GPU runtime before running the notebook.")

INPUT_ROOT = BUNDLE_ROOT / "input_text"
OUTPUT_ROOT = BUNDLE_ROOT / "output_embeddings"
REPORTS_ROOT = BUNDLE_ROOT / "reports"
CONFIG_PATH = BUNDLE_ROOT / "config" / "specter2_config.json"

if not INPUT_ROOT.exists():
    raise FileNotFoundError(INPUT_ROOT)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(CONFIG_PATH)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_SCRATCH.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    CONFIG = json.load(file)

MODEL_ID = CONFIG["specter2"]["base_model"]
ADAPTER_ID = CONFIG["specter2"]["adapter"]
MAX_LENGTH = int(CONFIG["specter2"]["max_length"])
EMBEDDING_DIM = int(CONFIG["specter2"]["embedding_dimension"])

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

print(f"Bundle: {BUNDLE_ROOT}")
print(f"GPU: {GPU_NAME}")
print(f"Model: {MODEL_ID}")
print(f"Adapter: {ADAPTER_ID}")
print(f"Maximum token length: {MAX_LENGTH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"FP16 inference: {USE_FP16}")

## Discover input shards

Each partition directory is processed as one resumable unit.

In [ ]:
def partition_value(partition_dir: Path, key: str) -> str:
    prefix = f"{key}="
    for part in partition_dir.parts:
        if part.startswith(prefix):
            return part[len(prefix):]
    raise ValueError(f"Missing partition key {key} in {partition_dir}")


PARQUET_FILES = sorted(INPUT_ROOT.rglob("*.parquet"))
PARTITION_DIRS = sorted({path.parent for path in PARQUET_FILES})

if not PARTITION_DIRS:
    raise RuntimeError(f"No Parquet files found under {INPUT_ROOT}")

shard_inventory = pd.DataFrame(
    [
        {
            "publication_year": int(partition_value(path, "publication_year")),
            "shard_id": int(partition_value(path, "shard_id")),
            "n_parquet_files": len(list(path.glob("*.parquet"))),
            "input_directory": str(path),
        }
        for path in PARTITION_DIRS
    ]
).sort_values(["publication_year", "shard_id"], ignore_index=True)

display(shard_inventory.head(20))
print(f"Input shards: {len(shard_inventory):,}")

## Load SPECTER2

The proximity adapter is loaded on top of the SPECTER2 base model. The first token representation is saved as the paper embedding.

In [ ]:
from adapters import AutoAdapterModel
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoAdapterModel.from_pretrained(MODEL_ID)
model.load_adapter(
    ADAPTER_ID,
    source="hf",
    load_as="specter2",
    set_active=True,
)
model.to(DEVICE)
model.eval()

if int(model.config.hidden_size) != EMBEDDING_DIM:
    raise ValueError(
        f"Configured embedding dimension {EMBEDDING_DIM} does not match "
        f"model hidden size {model.config.hidden_size}."
    )

print(f"Loaded model on {DEVICE}: {GPU_NAME}")

## Embedding functions

Outputs are written shard by shard:

- `index.parquet`
- `embeddings.npy`
- `metadata.json`

`metadata.json` is written last and acts as the completion marker.

In [ ]:
def output_directory(partition_dir: Path) -> Path:
    relative = partition_dir.relative_to(INPUT_ROOT)
    return OUTPUT_ROOT / relative


def completion_metadata(output_dir: Path) -> dict | None:
    metadata_path = output_dir / "metadata.json"
    index_path = output_dir / "index.parquet"
    embedding_path = output_dir / "embeddings.npy"

    if not metadata_path.exists():
        return None

    if not index_path.exists() or not embedding_path.exists():
        return None

    with metadata_path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)

    if metadata.get("status") != "completed":
        return None

    return metadata


def read_partition(partition_dir: Path) -> pd.DataFrame:
    parquet_files = sorted(partition_dir.glob("*.parquet"))

    frame = pd.concat(
        [
            pd.read_parquet(
                path,
                columns=[
                    "work_id",
                    "publication_year",
                    "title",
                    "abstract",
                    "language",
                    "text_hash",
                ],
            )
            for path in parquet_files
        ],
        ignore_index=True,
    )

    frame = frame.sort_values("work_id", kind="stable").reset_index(drop=True)

    if frame["work_id"].duplicated().any():
        duplicate_count = int(frame["work_id"].duplicated().sum())
        raise ValueError(
            f"{partition_dir} contains {duplicate_count:,} duplicate work IDs."
        )

    if frame["title"].isna().any() or frame["abstract"].isna().any():
        raise ValueError(f"{partition_dir} contains missing title or abstract values.")

    if (frame["title"].str.strip().str.len() == 0).any():
        raise ValueError(f"{partition_dir} contains empty titles.")

    if (frame["abstract"].str.strip().str.len() == 0).any():
        raise ValueError(f"{partition_dir} contains empty abstracts.")

    return frame


def atomic_copy(source: Path, destination: Path) -> None:
    temporary = destination.with_name(destination.name + ".tmp")
    shutil.copy2(source, temporary)
    os.replace(temporary, destination)


def embed_partition(partition_dir: Path) -> dict:
    started_at = time.time()
    year = int(partition_value(partition_dir, "publication_year"))
    shard_id = int(partition_value(partition_dir, "shard_id"))
    output_dir = output_directory(partition_dir)

    existing = completion_metadata(output_dir)
    if existing is not None and not OVERWRITE_COMPLETED:
        return {
            "publication_year": year,
            "shard_id": shard_id,
            "status": "skipped_completed",
            "n_rows": int(existing["n_output_rows"]),
            "elapsed_seconds": 0.0,
            "output_directory": str(output_dir),
        }

    frame = read_partition(partition_dir)
    n_rows = len(frame)

    scratch_dir = LOCAL_SCRATCH / f"year_{year}_shard_{shard_id}"
    if scratch_dir.exists():
        shutil.rmtree(scratch_dir)
    scratch_dir.mkdir(parents=True)

    scratch_embedding = scratch_dir / "embeddings.npy"
    scratch_index = scratch_dir / "index.parquet"
    scratch_metadata = scratch_dir / "metadata.json"

    embedding_memmap = np.lib.format.open_memmap(
        scratch_embedding,
        mode="w+",
        dtype=np.float32,
        shape=(n_rows, EMBEDDING_DIM),
    )

    total_batches = (n_rows + BATCH_SIZE - 1) // BATCH_SIZE

    batch_progress = tqdm(
        range(0, n_rows, BATCH_SIZE),
        total=total_batches,
        desc=f"{year}/{shard_id} batches",
        leave=False,
        position=1,
    )

    for start in batch_progress:
        end = min(start + BATCH_SIZE, n_rows)
        batch = frame.iloc[start:end]

        texts = (
            batch["title"].astype(str)
            + tokenizer.sep_token
            + batch["abstract"].astype(str)
        ).tolist()

        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
            return_token_type_ids=False,
        )
        encoded = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in encoded.items()
        }

        with torch.inference_mode():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_FP16,
            ):
                output = model(**encoded)
                batch_embeddings = output.last_hidden_state[:, 0, :]

        batch_embeddings = batch_embeddings.float().cpu().numpy()

        if not np.isfinite(batch_embeddings).all():
            raise ValueError(
                f"Non-finite embeddings detected in year {year}, shard {shard_id}."
            )

        embedding_memmap[start:end] = batch_embeddings

    embedding_memmap.flush()
    del embedding_memmap

    index = frame[
        [
            "work_id",
            "publication_year",
            "language",
            "text_hash",
        ]
    ].copy()
    index.insert(0, "row_id", np.arange(n_rows, dtype=np.int64))
    index.to_parquet(
        scratch_index,
        index=False,
        compression="zstd",
    )

    metadata = {
        "status": "completed",
        "publication_year": year,
        "shard_id": shard_id,
        "n_input_rows": n_rows,
        "n_output_rows": n_rows,
        "embedding_shape": [n_rows, EMBEDDING_DIM],
        "embedding_dtype": "float32",
        "base_model": MODEL_ID,
        "adapter": ADAPTER_ID,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "fp16_inference": USE_FP16,
        "gpu_name": GPU_NAME,
        "input_files": [
            str(path.relative_to(BUNDLE_ROOT))
            for path in sorted(partition_dir.glob("*.parquet"))
        ],
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": time.time() - started_at,
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "transformers_version": version("transformers"),
        "adapters_version": version("adapters"),
    }

    with scratch_metadata.open("w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)

    output_dir.mkdir(parents=True, exist_ok=True)
    atomic_copy(scratch_embedding, output_dir / "embeddings.npy")
    atomic_copy(scratch_index, output_dir / "index.parquet")
    atomic_copy(scratch_metadata, output_dir / "metadata.json")

    shutil.rmtree(scratch_dir)

    del frame
    del index
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "publication_year": year,
        "shard_id": shard_id,
        "status": "completed",
        "n_rows": n_rows,
        "elapsed_seconds": metadata["elapsed_seconds"],
        "output_directory": str(output_dir),
    }

## Generate embeddings

The outer progress bar tracks shards. The inner progress bar tracks GPU batches within the current shard.

In [ ]:
run_results = []

shard_progress = tqdm(
    PARTITION_DIRS,
    desc="Embedding shards",
    position=0,
)

for partition_dir in shard_progress:
    year = partition_value(partition_dir, "publication_year")
    shard_id = partition_value(partition_dir, "shard_id")
    shard_progress.set_postfix(year=year, shard=shard_id)

    result = embed_partition(partition_dir)
    run_results.append(result)

run_results_frame = pd.DataFrame(run_results).sort_values(
    ["publication_year", "shard_id"],
    ignore_index=True,
)

run_results_frame.to_parquet(
    REPORTS_ROOT / "embedding_run_results.parquet",
    index=False,
    compression="zstd",
)

display(run_results_frame["status"].value_counts().rename_axis("status").to_frame("n_shards"))
print(f"Processed or validated shards: {len(run_results_frame):,}")

## Validate all outputs

This cell verifies the completion marker, index row count, NumPy shape, embedding dimension, and file presence for every expected shard.

In [ ]:
validation_rows = []

for partition_dir in tqdm(PARTITION_DIRS, desc="Validating outputs"):
    year = int(partition_value(partition_dir, "publication_year"))
    shard_id = int(partition_value(partition_dir, "shard_id"))
    output_dir = output_directory(partition_dir)

    metadata_path = output_dir / "metadata.json"
    index_path = output_dir / "index.parquet"
    embedding_path = output_dir / "embeddings.npy"

    metadata_exists = metadata_path.exists()
    index_exists = index_path.exists()
    embedding_exists = embedding_path.exists()

    status = None
    metadata_rows = None
    index_rows = None
    embedding_rows = None
    embedding_columns = None

    if metadata_exists:
        with metadata_path.open("r", encoding="utf-8") as file:
            metadata = json.load(file)
        status = metadata.get("status")
        metadata_rows = int(metadata.get("n_output_rows", -1))

    if index_exists:
        index_rows = int(pq.ParquetFile(index_path).metadata.num_rows)

    if embedding_exists:
        array = np.load(embedding_path, mmap_mode="r")
        embedding_rows = int(array.shape[0])
        embedding_columns = int(array.shape[1]) if array.ndim == 2 else None
        del array

    valid = (
        metadata_exists
        and index_exists
        and embedding_exists
        and status == "completed"
        and metadata_rows == index_rows
        and index_rows == embedding_rows
        and embedding_columns == EMBEDDING_DIM
    )

    validation_rows.append(
        {
            "publication_year": year,
            "shard_id": shard_id,
            "valid": valid,
            "status": status,
            "metadata_rows": metadata_rows,
            "index_rows": index_rows,
            "embedding_rows": embedding_rows,
            "embedding_columns": embedding_columns,
            "output_directory": str(output_dir),
        }
    )

validation_frame = pd.DataFrame(validation_rows).sort_values(
    ["publication_year", "shard_id"],
    ignore_index=True,
)

validation_frame.to_parquet(
    REPORTS_ROOT / "embedding_validation.parquet",
    index=False,
    compression="zstd",
)

RUN_COMPLETE = bool(validation_frame["valid"].all())
TOTAL_EMBEDDINGS = int(validation_frame["embedding_rows"].sum())
TOTAL_OUTPUT_BYTES = sum(
    path.stat().st_size
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

summary = {
    "run_complete": RUN_COMPLETE,
    "expected_shards": len(PARTITION_DIRS),
    "valid_shards": int(validation_frame["valid"].sum()),
    "total_embeddings": TOTAL_EMBEDDINGS,
    "embedding_dimension": EMBEDDING_DIM,
    "output_bytes": TOTAL_OUTPUT_BYTES,
    "output_gib": TOTAL_OUTPUT_BYTES / (1024 ** 3),
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
}

with (REPORTS_ROOT / "embedding_summary.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

display(validation_frame["valid"].value_counts().rename_axis("valid").to_frame("n_shards"))
print(json.dumps(summary, indent=2))

if not RUN_COMPLETE:
    invalid = validation_frame.loc[~validation_frame["valid"]]
    display(invalid)
    raise RuntimeError("Embedding output validation failed.")

## Terminate the Colab runtime

When `AUTO_TERMINATE = True`, the runtime is released only after every expected shard passes validation.

In [ ]:
if not RUN_COMPLETE:
    raise RuntimeError("The runtime will not terminate because validation did not complete.")

if AUTO_TERMINATE:
    print(
        f"All {len(PARTITION_DIRS):,} shards passed validation. "
        f"Terminating the Colab runtime in {TERMINATION_DELAY_SECONDS} seconds."
    )
    os.sync()
    time.sleep(TERMINATION_DELAY_SECONDS)

    from google.colab import runtime

    runtime.unassign()
else:
    print("AUTO_TERMINATE is False. The runtime remains connected.")